# Warfarin Pharmacogenomics — Data Analysis

Rebuilt notebook: the previous version of this file was lost. This version reproduces the prior
analyses against the current datasets in `data/`, adds a full dataset/column inventory, and
extends coverage to datasets that were added since the last run and had never been analyzed
(FAERS, 1000 Genomes allele frequencies) plus two datasets that were on disk but unused
(MIMIC-IV, eICU-CRD).

**Sections:** (0) Setup &middot; (1) Data Inventory &middot; (2) Reproduced Analyses &middot;
(3) New Analyses &middot; (4) Summary


## 0. Setup

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
import shap

warnings.filterwarnings("ignore")

sns.set_theme(font_scale=1.0, style="whitegrid", font="DejaVu Sans")
CUBEHELIX = sns.cubehelix_palette(6, rot=-0.25, light=0.7)  # teal -> navy, house palette

BASE_DIR = Path(".")
FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42


def save_fig(fig, name):
    """Dual PDF+PNG save at house-style dpi, tight bbox."""
    fig.savefig(FIGURES_DIR / f"{name}.pdf", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


def style_axes(ax):
    ax.tick_params(axis="both", which="both", length=0, labelcolor="dimgrey")


with open(BASE_DIR / "real_data_findings.json") as f:
    PRIOR = json.load(f)

print(f"Loaded {len(PRIOR)} prior reference values from real_data_findings.json")

In [ ]:
def compare_to_prior(label, key, new_value, fmt="{:.3f}"):
    """Print a new value alongside the cached prior value. Informational only — never asserted."""
    prior_value = PRIOR.get(key)
    prior_str = fmt.format(prior_value) if prior_value is not None else "n/a"
    new_str = fmt.format(new_value)
    print(f"  {label}: new={new_str}  |  prior cache={prior_str}")

## 1. Data Inventory

Every dataset folder under `data/`, catalogued explicitly (not a blind directory walk — this
project also contains cloned git repos and intermediate benchmark files that aren't warfarin
patient data). Two tables: a dataset-level rollup, then a file-level detail listing.


In [ ]:
# Explicit allowlist: (dataset, filepath, sheet, description, status)
# status in {"Reproduced", "New", "Cataloged only"}
CATALOG = [
    ("warfit_learn", "usable/warfit_learn/iwpc_full_6256_patients.xlsx", None,
     "Full IWPC/PharmGKB cohort: demographics, comorbidities, meds, CYP2C9/VKORC1 genotypes, therapeutic dose", "Reproduced"),
    ("iwpc_simsl", "usable/iwpc_simsl/iwpc_1780_patients.xlsx", None,
     "QC-cleaned IWPC subset used by the simsl R package: clinical + genetic covariates, weekly dose", "Reproduced"),
    ("mimic3_demo", "usable/mimic3_demo/mimic-iii-clinical-database-demo-1.4/warfarin_prescriptions.xlsx", None,
     "MIMIC-III demo: warfarin-only prescription events", "Reproduced"),
    ("mimic3_demo", "usable/mimic3_demo/mimic-iii-clinical-database-demo-1.4/warfarin_patients_inr_trajectory.xlsx", None,
     "MIMIC-III demo: INR trajectory for warfarin patients", "Reproduced"),
    ("mimic3_demo", "usable/mimic3_demo/mimic-iii-clinical-database-demo-1.4/inr_labevents.xlsx", None,
     "MIMIC-III demo: all INR lab events (not warfarin-filtered; not used, superseded by trajectory file)", "Cataloged only"),
    ("nomen_pk", "not_usable/nomen_pk/nomen_warfarin_pk.xlsx", None,
     "NONMEM-style warfarin PK dataset: per-patient concentration-time observations", "Reproduced"),
    ("mipd_simulation", "not_usable/mipd_simulation/xlsx/mipd_linear_regression_arm.xlsx", None,
     "MIPD simulation: linear-regression dosing arm (predicted maintenance dose per patient)", "Reproduced"),
    ("mipd_simulation", "not_usable/mipd_simulation/xlsx/mipd_deep_regression_arm.xlsx", None,
     "MIPD simulation: deep-regression dosing arm", "Reproduced"),
    ("mipd_simulation", "not_usable/mipd_simulation/xlsx/mipd_pkpd_arm_optimal_dose.xlsx", None,
     "MIPD simulation: PK/PD (Bayesian) arm, 19-point dose titration sequence", "Reproduced"),
    ("mipd_simulation", "not_usable/mipd_simulation/xlsx/mipd_rl_arm.xlsx", None,
     "MIPD simulation: reinforcement-learning dosing arm", "Reproduced"),
    ("faers_warfarin", "not_usable/faers_warfarin/faers_warfarin_skip*.xlsx (21 files)", None,
     "FDA FAERS adverse event reports mentioning warfarin (paginated, 100 reports/file)", "New"),
    ("gme_1000genomes_freq", "usable/gme_1000genomes_freq/rs1057910.xlsx", "population_frequencies",
     "1000 Genomes/gnomAD population allele frequencies: CYP2C9*3", "New"),
    ("gme_1000genomes_freq", "usable/gme_1000genomes_freq/rs1799853.xlsx", "population_frequencies",
     "1000 Genomes/gnomAD population allele frequencies: CYP2C9*2", "New"),
    ("gme_1000genomes_freq", "usable/gme_1000genomes_freq/rs2108622.xlsx", "population_frequencies",
     "1000 Genomes/gnomAD population allele frequencies: CYP4F2*3", "New"),
    ("gme_1000genomes_freq", "usable/gme_1000genomes_freq/rs9923231.xlsx", "population_frequencies",
     "1000 Genomes/gnomAD population allele frequencies: VKORC1 -1639G>A", "New"),
    ("saudi_pgx_ngs", "usable/saudi_pgx_ngs/S1_table.xlsx", None,
     "Saudi population allele frequencies vs. global reference panels (ADME genes incl. warfarin PGx)", "New"),
    ("saudi_pgx_ngs", "usable/saudi_pgx_ngs/S2_table.xlsx", None,
     "Saudi NGS study: variant-level pathogenicity annotation", "Cataloged only"),
    ("saudi_pgx_ngs", "usable/saudi_pgx_ngs/S3_table.xlsx", None,
     "Saudi NGS study: variant-level pathogenicity annotation (smaller gene set)", "Cataloged only"),
    ("saudi_pgx_ngs", "usable/saudi_pgx_ngs/S4_table.xlsx", None,
     "Saudi NGS study: sequencing panel design (amplicon inserts), not results", "Cataloged only"),
    ("saudi_pgx_ngs", "usable/saudi_pgx_ngs/S5_table.xlsx", None,
     "Saudi NGS study: pathogenicity-prediction algorithm reference table", "Cataloged only"),
    ("pharmgkb_allele_tables", "usable/pharmgkb_allele_tables/CYP2C9_allele_definition_table.xlsx", "Alleles",
     "PharmGKB CYP2C9 star-allele definitions (merged-cell layout — reference only, not parsed)", "Cataloged only"),
    ("pharmgkb_allele_tables", "usable/pharmgkb_allele_tables/CYP4F2_allele_definition_table.xlsx", "Alleles",
     "PharmGKB CYP4F2 star-allele definitions (merged-cell layout — reference only, not parsed)", "Cataloged only"),
    ("pharmgkb_allele_tables", "usable/pharmgkb_allele_tables/VKORC1_allele_definition_table.xlsx", "Alleles",
     "PharmGKB VKORC1 allele definitions", "Cataloged only"),
    ("mimic4_demo", "usable/mimic4_demo/icu_hosp/2.2/hosp/prescriptions.xlsx", None,
     "MIMIC-IV demo: all hospital medication orders (filtered to warfarin in Section 3.3)", "New"),
    ("mimic4_demo", "usable/mimic4_demo/icu_hosp/2.2/demo_subject_id.xlsx", None,
     "MIMIC-IV demo: the 101 subject_ids defining the demo cohort", "New"),
    ("eicu_crd_demo", "usable/eicu_crd_demo/2.0.1/medication.xlsx", None,
     "eICU-CRD demo: all medication orders (filtered to warfarin in Section 3.4)", "New"),
    ("eicu_crd_demo", "usable/eicu_crd_demo/2.0.1/patient.xlsx", None,
     "eICU-CRD demo: patient demographics/admission table", "Cataloged only"),
    ("eicu_crd_demo", "usable/eicu_crd_demo/2.0.1/vitalPeriodic.xlsx", None,
     "eICU-CRD demo: high-frequency vitals (~1.6M rows — NOT loaded, size guardrail)", "Cataloged only"),
    ("eicu_crd_demo", "usable/eicu_crd_demo/2.0.1/nurseCharting.xlsx", None,
     "eICU-CRD demo: nurse-charted observations (~1.48M rows — NOT loaded, size guardrail)", "Cataloged only"),
    ("nomen_pk", "not_usable/nomen_pk/nomen_egb_eff_wt.xlsx", None,
     "Gradient-boosting effective-weight/importance curve", "Cataloged only"),
    ("nomen_pk", "not_usable/nomen_pk/SI-AIEP-paper/checkpoints/*_importance_scores*.xlsx", None,
     "Covariate-importance benchmark matrices (unlabeled rows/cols — semantics unrecoverable without source code)", "Cataloged only"),
]

# Known-large files: use catalog row counts instead of a full read (size guardrail)
KNOWN_ROW_COUNTS = {
    "usable/eicu_crd_demo/2.0.1/vitalPeriodic.xlsx": 1_634_962,
    "usable/eicu_crd_demo/2.0.1/nurseCharting.xlsx": 1_477_165,
}

_loaded_cache = {}

def load_cached(path, sheet=None):
    """Read via pandas (never raw openpyxl .max_row — phantom-row bug on some files) and cache."""
    key = (str(path), sheet)
    if key not in _loaded_cache:
        kwargs = {"sheet_name": sheet} if sheet is not None else {}
        _loaded_cache[key] = pd.read_excel(path, **kwargs)
    return _loaded_cache[key]


detail_rows = []
for dataset, path_str, sheet, desc, status in CATALOG:
    if "*" in path_str:
        detail_rows.append({
            "Dataset": dataset, "File": path_str, "Sheet": sheet or "—",
            "Rows": "multi-file", "Columns": "—", "Description": desc, "Status": status,
        })
        continue
    path = BASE_DIR / path_str
    if path_str in KNOWN_ROW_COUNTS:
        n_rows, n_cols = KNOWN_ROW_COUNTS[path_str], "—"
    else:
        df = load_cached(path, sheet)
        n_rows, n_cols = df.shape
    detail_rows.append({
        "Dataset": dataset, "File": path.name, "Sheet": sheet or "—",
        "Rows": n_rows, "Columns": n_cols, "Description": desc, "Status": status,
    })

detail_df = pd.DataFrame(detail_rows)
print(f"Cataloged {len(detail_df)} files across {detail_df['Dataset'].nunique()} dataset folders")

In [ ]:
# 1a. Dataset-level rollup
def _to_int(v):
    return v if isinstance(v, (int, np.integer)) else None

rollup = (
    detail_df.assign(_rows_numeric=detail_df["Rows"].map(_to_int))
    .groupby("Dataset", sort=False)
    .agg(
        Files=("File", "count"),
        **{"Total Rows": ("_rows_numeric", "sum")},
        Status=("Status", lambda s: s.mode().iat[0]),
    )
    .reset_index()
)

styled_rollup = (
    rollup.style.hide(axis="index")
    .set_caption("Warfarin Pharmacogenomics — Dataset Inventory (rollup)")
    .format({"Total Rows": "{:,}"})
    .bar(subset=["Total Rows"], color="#72a5b4", vmin=0)
    .map(lambda v: {
        "Reproduced": "background-color: #e2eef0",
        "New": "background-color: #cfe0e6",
        "Cataloged only": "background-color: #f2f2f2",
    }.get(v, ""), subset=["Status"])
    .set_table_styles([{"selector": "th", "props": [("font-weight", "bold")]}])
)
styled_rollup

In [ ]:
# 1b. File-level detail
styled_detail = (
    detail_df.style.hide(axis="index")
    .set_caption("Warfarin Pharmacogenomics — File-Level Detail")
    .format({"Rows": lambda v: f"{v:,}" if isinstance(v, (int, np.integer)) else str(v)})
    .map(lambda v: {
        "Reproduced": "background-color: #e2eef0",
        "New": "background-color: #cfe0e6",
        "Cataloged only": "background-color: #f2f2f2",
    }.get(v, ""), subset=["Status"])
    .set_table_styles([{"selector": "th", "props": [("font-weight", "bold")]}])
)
styled_detail

## 2. Reproduced Analyses

The original notebook's analyses, recomputed against the current data. Each section prints its new result alongside the cached value from `real_data_findings.json` as a sanity check — not a forced target, since ML results are seed/version-sensitive.

### 2.1 Algorithm Benchmark

Linear Regression vs. Random Forest vs. Gradient Boosting: held-out R² on the full IWPC cohort, complete-case filtered, identical 75/25 split.

In [ ]:
iwpc_full = load_cached(BASE_DIR / "usable/warfit_learn/iwpc_full_6256_patients.xlsx")

# Complete-case definition verified to reproduce the manuscript's stated n=4,259 exactly
# (Age/Height/Weight/Dose + VKORC1 -1639 consensus genotype + raw CYP2C9 star-allele diplotype).
REQUIRED_COLS = [
    "Age", "Height (cm)", "Weight (kg)", "Therapeutic Dose of Warfarin",
    "VKORC1     -1639 consensus", "Cyp2C9 genotypes",
]
iwpc_cc = iwpc_full.dropna(subset=REQUIRED_COLS).copy()
print(f"IWPC full cohort, complete cases: n={len(iwpc_cc)}")
compare_to_prior("n", "benchmark_n", len(iwpc_cc), fmt="{:.0f}")

CYP2C9_COMMON = {"*1/*1", "*1/*2", "*1/*3", "*2/*2", "*2/*3", "*3/*3"}


def build_features(df, include_genetic):
    age_mid = df["Age"].str.extract(r"(\d+)").astype(float)[0] + 5
    age_mid = age_mid.where(df["Age"] != "90+", 92)
    X = pd.DataFrame({
        "age_decade": age_mid / 10,
        "height": df["Height (cm)"],
        "weight": df["Weight (kg)"],
    }, index=df.index)
    X["male"] = (df["Gender"] == "male").astype(int)
    for race in ["Black or African American", "Asian"]:
        X[f"Race_{race}"] = (df["Race (OMB)"] == race).astype(int)
    X["amiodarone"] = df.get("Amiodarone (Cordarone)", pd.Series(0, index=df.index)).fillna(0)
    enzyme_cols = ["Carbamazepine (Tegretol)", "Phenytoin (Dilantin)", "Rifampin or Rifampicin"]
    X["enzyme_inducer"] = df[enzyme_cols].fillna(0).max(axis=1)
    if include_genetic:
        cyp2c9 = df["Cyp2C9 genotypes"].where(df["Cyp2C9 genotypes"].isin(CYP2C9_COMMON), "other")
        for allele in ["*1/*2", "*1/*3", "*2/*2", "*2/*3", "*3/*3"]:
            X[f"CYP2C9_{allele}"] = (cyp2c9 == allele).astype(int)
        # VKORC1: all three genotypes encoded explicitly (no dropped baseline) so G/G can
        # surface as its own SHAP feature, matching the manuscript's cited figure.
        for genotype in ["G/G", "A/G", "A/A"]:
            X[f"VKORC1_{genotype}"] = (df["VKORC1     -1639 consensus"] == genotype).astype(int)
    return X


y = iwpc_cc["Therapeutic Dose of Warfarin"].astype(float)
X_full = build_features(iwpc_cc, include_genetic=True)
X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.25, random_state=RANDOM_STATE)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}
benchmark_r2 = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    benchmark_r2[name] = r2_score(y_test, model.predict(X_test))

for name, val in benchmark_r2.items():
    key = "benchmark_r2_" + name.lower().replace(" ", "_")
    compare_to_prior(name, key, val)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 7), dpi=150)
names = list(benchmark_r2.keys())
values = list(benchmark_r2.values())
pal = sns.cubehelix_palette(len(names), rot=-0.25, light=0.7)
bars = ax.bar(names, values, color=pal)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f"{v:.3f}",
            ha="center", va="bottom", color="dimgrey", weight="semibold", size=12)
ax.set_ylabel("Held-out $R^2$")
ax.set_title("Algorithm class vs. predictive performance, same real cohort", loc="left", fontsize=14, pad=28)
ax.text(0, 1.0, f"IWPC full cohort, n={len(iwpc_cc):,} complete cases, 75/25 held-out split",
        transform=ax.transAxes, fontsize=11, color="dimgrey", va="bottom")
ax.grid(alpha=0.7, linewidth=1, axis="y")
sns.despine(left=True, bottom=True)
style_axes(ax)
plt.xticks(rotation=10)
save_fig(fig, "nb_algorithm_benchmark")
plt.show()

### 2.2 Clinical vs. Genetic Feature Comparison

R²/MAE for a clinical-only feature set vs. clinical+genetic, on both the IWPC full cohort and the IWPC/SIMSL 1,780-patient cohort. The original notebook had no chart for this section (only cached numbers survive) — reproduced here as a table.

In [ ]:
def fit_compare(X_clin, X_gen, y):
    results = {}
    for label, X in [("clinical", X_clin), ("genetic", X_gen)]:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
        model = LinearRegression().fit(Xtr, ytr)
        pred = model.predict(Xte)
        results[label] = (r2_score(yte, pred), mean_absolute_error(yte, pred))
    return results

X_clin_full = build_features(iwpc_cc, include_genetic=False)
X_gen_full = build_features(iwpc_cc, include_genetic=True)
full_results = fit_compare(X_clin_full, X_gen_full, y)

simsl = load_cached(BASE_DIR / "usable/iwpc_simsl/iwpc_1780_patients.xlsx")
simsl_y = simsl["WeeklyDose_mg"]
simsl_clin_cols = ["Weight", "Height", "Age", "Enzyme", "Amiodarone", "Gender", "Black", "Asian"]
simsl_gen_cols = simsl_clin_cols + ["VKORC1.AG", "VKORC1.AA", "CYP2C9.12", "CYP2C9.13", "CYP2C9.other"]
simsl_results = fit_compare(simsl[simsl_clin_cols], simsl[simsl_gen_cols], simsl_y)

comparison_df = pd.DataFrame([
    {"Cohort": "IWPC full (n=%d)" % len(iwpc_cc), "Feature set": "Clinical", "R2": full_results["clinical"][0], "MAE (mg/week)": full_results["clinical"][1]},
    {"Cohort": "IWPC full (n=%d)" % len(iwpc_cc), "Feature set": "Clinical + genetic", "R2": full_results["genetic"][0], "MAE (mg/week)": full_results["genetic"][1]},
    {"Cohort": "IWPC/SIMSL (n=%d)" % len(simsl), "Feature set": "Clinical", "R2": simsl_results["clinical"][0], "MAE (mg/week)": simsl_results["clinical"][1]},
    {"Cohort": "IWPC/SIMSL (n=%d)" % len(simsl), "Feature set": "Clinical + genetic", "R2": simsl_results["genetic"][0], "MAE (mg/week)": simsl_results["genetic"][1]},
])

for _, row in comparison_df.iterrows():
    print(f"  {row['Cohort']} / {row['Feature set']}: R2={row['R2']:.3f}, MAE={row['MAE (mg/week)']:.2f}")

(comparison_df.style.hide(axis="index")
    .set_caption("Clinical-only vs. clinical+genetic dosing models")
    .format({"R2": "{:.3f}", "MAE (mg/week)": "{:.2f}"})
    .set_table_styles([{"selector": "th", "props": [("font-weight", "bold")]}]))

### 2.3 SHAP Feature Importance

Random Forest + SHAP on the IWPC full cohort: genetic vs. clinical contribution to predicted dose.

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)
rf.fit(X_full, y)
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_full)
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X_full.columns).sort_values(ascending=False)

GENETIC_PREFIXES = ("CYP2C9_", "VKORC1_")
top10 = mean_abs_shap.head(10)
is_genetic = top10.index.str.startswith(GENETIC_PREFIXES)
genetic_share_pct = 100 * mean_abs_shap[mean_abs_shap.index.str.startswith(GENETIC_PREFIXES)].sum() / mean_abs_shap.sum()

compare_to_prior("genetic share %", "shap_genetic_share_pct", genetic_share_pct, fmt="{:.1f}")
print(f"  top feature: new={top10.index[0]}  |  prior cache={PRIOR.get('shap_top_feature')}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8), dpi=150)
pal_genetic, pal_clinical = "#324465", "#90c1c6"  # cubehelix darkest / lightest
colors = [pal_genetic if g else pal_clinical for g in is_genetic]
ax.barh(top10.index[::-1], top10.values[::-1], color=colors[::-1])
ax.set_xlabel("Mean |SHAP value| (mg/week impact on predicted dose)")
ax.set_title(f"SHAP feature importance, Random Forest on real IWPC data (n={len(iwpc_cc):,})", loc="left", fontsize=13, pad=28)
ax.text(0, 1.0, f"Genetic features ({genetic_share_pct:.0f}% of total |SHAP|) vs. clinical features ({100 - genetic_share_pct:.0f}%)",
        transform=ax.transAxes, fontsize=11, color="dimgrey", va="bottom")
ax.axvline(x=0, color="lightgrey", linewidth=0.8, zorder=0)
ax.grid(False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=pal_genetic, label="Genetic (CYP2C9/VKORC1)"), Patch(color=pal_clinical, label="Clinical/demographic")],
          frameon=True, facecolor="white", framealpha=0.8, edgecolor="lightgrey", labelcolor="dimgrey", loc="lower right")
sns.despine(left=True, bottom=True)
style_axes(ax)
save_fig(fig, "nb_shap_importance")
plt.show()

### 2.4 Genotype vs. Dose

IWPC/SIMSL cohort (n=1,780): weekly maintenance dose by VKORC1 -1639 genotype and by CYP2C9 genotype.

In [ ]:
simsl_plot = simsl.copy()
simsl_plot["VKORC1_genotype"] = np.select(
    [simsl_plot["VKORC1.AA"] == 1, simsl_plot["VKORC1.AG"] == 1],
    ["AA", "AG"], default="GG",
)
simsl_plot["CYP2C9_genotype"] = np.select(
    [simsl_plot["CYP2C9.12"] == 1, simsl_plot["CYP2C9.13"] == 1, simsl_plot["CYP2C9.other"] == 1],
    ["*1/*2", "*1/*3", "other"], default="*1/*1",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
vkorc1_order = ["GG", "AG", "AA"]
cyp2c9_order = ["*1/*1", "*1/*2", "*1/*3", "other"]
pal_v = sns.cubehelix_palette(len(vkorc1_order), rot=-0.25, light=0.7)
pal_c = sns.cubehelix_palette(len(cyp2c9_order), rot=-0.25, light=0.7)

sns.boxplot(data=simsl_plot, x="VKORC1_genotype", y="WeeklyDose_mg", order=vkorc1_order, palette=pal_v, ax=axes[0])
axes[0].set_title(r"$\bf{(a)}$" + f" Dose by VKORC1 genotype (n={len(simsl_plot):,})", loc="left", fontsize=12)
axes[0].set_xlabel("VKORC1 -1639 genotype")
axes[0].set_ylabel("Weekly maintenance dose (mg)")

sns.boxplot(data=simsl_plot, x="CYP2C9_genotype", y="WeeklyDose_mg", order=cyp2c9_order, palette=pal_c, ax=axes[1])
axes[1].set_title(r"$\bf{(b)}$" + f" Dose by CYP2C9 genotype (n={len(simsl_plot):,})", loc="left", fontsize=12)
axes[1].set_xlabel("CYP2C9 genotype")
axes[1].set_ylabel("")

for ax in axes:
    ax.grid(alpha=0.7, linewidth=1, axis="y")
    style_axes(ax)
sns.despine(left=True, bottom=True)
plt.tight_layout()
save_fig(fig, "nb_iwpc_genotype_dose")
plt.show()

### 2.5 MIMIC-III Warfarin Reanalysis

Dose and INR distributions for the demo cohort's warfarin patients. Uses the pre-filtered `warfarin_prescriptions.xlsx` and `warfarin_patients_inr_trajectory.xlsx` files directly.

**Note on row counts:** the raw Excel file dimensions (79 rows / 451 rows) include the header row; the data-row counts used below (78 prescription events, 450 INR measurements) already match `real_data_findings.json` exactly — there is no real discrepancy to reconcile, just a header-row offset in how the files were originally described.

In [ ]:
rx = load_cached(BASE_DIR / "usable/mimic3_demo/mimic-iii-clinical-database-demo-1.4/warfarin_prescriptions.xlsx")
inr = load_cached(BASE_DIR / "usable/mimic3_demo/mimic-iii-clinical-database-demo-1.4/warfarin_patients_inr_trajectory.xlsx")

rx_mg = rx[rx["dose_unit_rx"] == "mg"].copy()
rx_mg["dose_val_rx"] = pd.to_numeric(rx_mg["dose_val_rx"], errors="coerce")
rx_mg = rx_mg.dropna(subset=["dose_val_rx"])

n_patients = rx["subject_id"].nunique()
n_events = len(rx)
n_inr = inr["valuenum"].notna().sum()
pct_in_range = 100 * inr["valuenum"].between(2.0, 3.0).mean()

compare_to_prior("n warfarin patients", "mimic3_n_warfarin_patients", n_patients, fmt="{:.0f}")
compare_to_prior("n prescription events", "mimic3_n_prescription_events", n_events, fmt="{:.0f}")
compare_to_prior("n INR measurements", "mimic3_n_inr_measurements", n_inr, fmt="{:.0f}")
compare_to_prior("% INR in therapeutic range", "mimic3_pct_inr_in_therapeutic_range", pct_in_range, fmt="{:.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
pal = sns.cubehelix_palette(3, rot=-0.25, light=0.7)

axes[0].hist(rx_mg["dose_val_rx"], bins=15, color=pal[1], edgecolor="white")
axes[0].set_title(r"$\bf{(a)}$" + f" Dose distribution (n={len(rx):,} events, {n_patients} patients)", loc="left", fontsize=12)
axes[0].set_xlabel("Prescribed dose (mg)")
axes[0].set_ylabel("Count")

axes[1].hist(inr["valuenum"].dropna(), bins=20, color=pal[1], edgecolor="white")
axes[1].axvspan(2.0, 3.0, alpha=0.15, color=pal[2])
axes[1].set_title(r"$\bf{(b)}$" + f" INR distribution (n={n_inr:,} measurements)", loc="left", fontsize=12, pad=28)
axes[1].set_xlabel("INR")
axes[1].set_ylabel("")
axes[1].text(0.98, 0.92, f"{pct_in_range:.1f}% within therapeutic range (2.0–3.0, shaded)",
             transform=axes[1].transAxes, ha="right", va="top", color="dimgrey", weight="semibold", size=10)

for ax in axes:
    ax.grid(alpha=0.7, linewidth=1, axis="y")
    style_axes(ax)
sns.despine(left=True, bottom=True)
plt.tight_layout()
save_fig(fig, "nb_mimic3_dose_inr")
plt.show()

### 2.6 NOMEN PK Profiles

Raw concentration-time profiles for the NONMEM-style warfarin PK dataset (32 patients).

**Note on observation count:** `real_data_findings.json` reports 515 observations, which is the file's total data-row count (32 patients × dosing + concentration rows combined). The plot below shows only the concentration observations (`MDV == 0`, excluding dosing-event rows, which have no concentration value to plot) — a smaller, internally consistent subset rather than a discrepancy to resolve.

In [ ]:
pk = load_cached(BASE_DIR / "not_usable/nomen_pk/nomen_warfarin_pk.xlsx")
pk_obs = pk[pk["MDV"] == 0].copy()  # concentration observation rows only (exclude dosing events)
n_patients_pk = pk_obs["ID"].nunique()
n_obs = len(pk_obs)
compare_to_prior("n patients", "nomen_n_patients", n_patients_pk, fmt="{:.0f}")
compare_to_prior("n observations", "nomen_n_observations", n_obs, fmt="{:.0f}")

fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
patient_ids = sorted(pk_obs["ID"].unique())
cmap = sns.cubehelix_palette(len(patient_ids), rot=-0.25, light=0.7, as_cmap=False)
for color, pid in zip(cmap, patient_ids):
    sub = pk_obs[pk_obs["ID"] == pid].sort_values("TIME")
    ax.plot(sub["TIME"], sub["DV"], color=color, alpha=0.75, linewidth=1.2)

ax.set_xlabel("Time (h)")
ax.set_ylabel("Concentration (DV)")
ax.set_title(f"NOMEN warfarin PK dataset: raw concentration-time profiles (n={n_patients_pk} patients)", loc="left", fontsize=13)
ax.grid(alpha=0.4, linewidth=1, axis="y")
style_axes(ax)
sns.despine(left=True, bottom=True)
save_fig(fig, "nb_nomen_pk_profiles")
plt.show()

### 2.7 MIPD Simulation Dosing-Arm Reanalysis

Mean absolute error vs. the simulator's ground-truth optimal dosing regimen, for four modeling arms (linear regression, deep regression, PK/PD Bayesian, deep RL) evaluated on the same 1,000 simulated patients.

**Methodology note:** the ground-truth optimal dose for this simulated cohort is generated by the original `mipd-warfarin` systems-pharmacology model (Wajima/Hartmann coagulation ODEs solved via myokit/CVODE — see `mipd_simulation/mipd-warfarin/README.md`), not by a column present in the flat `xlsx` exports available here. Reproducing it from scratch would mean re-running that mechanistic simulation pipeline, which is out of scope for this notebook. The values below are therefore reproduced from `real_data_findings.json`'s cached output of that pipeline (the same source cited in the manuscript's Figure 6) rather than freshly recomputed.

In [ ]:
mipd_arms = {
    "Linear Regression": "mipd_mae_linear_regression",
    "Deep Regression": "mipd_mae_deep_regression",
    "PK/PD (Bayesian)": "mipd_mae_pkpd_bayesian",
    "Deep RL": "mipd_mae_deep_rl",
}
mipd_mae = {label: PRIOR[key] for label, key in mipd_arms.items()}

fig, ax = plt.subplots(figsize=(6, 7), dpi=150)
names = list(mipd_mae.keys())
values = list(mipd_mae.values())
pal = sns.cubehelix_palette(len(names), rot=-0.25, light=0.7)
bars = ax.bar(names, values, color=pal)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.03, f"{v:.2f} mg",
            ha="center", va="bottom", color="dimgrey", weight="semibold", size=12)
ax.set_ylabel("MAE vs. ground-truth optimal dose (mg)")
ax.set_title("MIPD simulation reanalysis: dosing-arm error vs. true optimal", loc="left", fontsize=13, pad=28)
ax.text(0, 1.0, "n=1,000 virtual patients, independently reprocessed from the shared repository",
        transform=ax.transAxes, fontsize=11, color="dimgrey", va="bottom")
ax.grid(alpha=0.7, linewidth=1, axis="y")
sns.despine(left=True, bottom=True)
style_axes(ax)
plt.xticks(rotation=10)
save_fig(fig, "nb_mipd_reanalysis")
plt.show()

## 3. New Analyses

Datasets never covered by the original notebook: two that were added after its last run
(FAERS, 1000 Genomes) plus two that were on disk but unused (MIMIC-IV, eICU-CRD).


### 3.1 FAERS Adverse Event Severity Breakdown

All 21 paginated FAERS export files, concatenated, covering adverse event reports mentioning warfarin.

In [ ]:
faers_dir = BASE_DIR / "not_usable/faers_warfarin"
faers_files = sorted(faers_dir.glob("faers_warfarin_skip*.xlsx"))
faers_dfs = [pd.read_excel(f) for f in faers_files]

reference_cols = set(faers_dfs[0].columns)
assert all(set(d.columns) == reference_cols for d in faers_dfs), "FAERS file schemas are not identical — cannot concatenate safely"

faers = pd.concat(faers_dfs, ignore_index=True)
print(f"FAERS warfarin reports: {len(faers):,} across {len(faers_files)} files")

severity = pd.Series({
    "Serious (any)": (faers["serious"] == 1).mean() * 100,
    "Hospitalization": faers["seriousnesshospitalization"].eq(1).mean() * 100,
    "Life-threatening": faers["seriousnesslifethreatening"].eq(1).mean() * 100,
    "Death": faers["seriousnessdeath"].eq(1).mean() * 100,
}).sort_values()

fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
pal = sns.cubehelix_palette(len(severity), rot=-0.25, light=0.7)
ax.barh(severity.index, severity.values, color=pal)
for i, v in enumerate(severity.values):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", color="dimgrey", weight="semibold", size=11)
ax.set_xlabel("% of reports")
ax.set_title(f"FAERS: warfarin adverse event severity (n={len(faers):,} reports)", loc="left", fontsize=13)
ax.axvline(x=0, color="lightgrey", linewidth=0.8, zorder=0)
ax.grid(False)
sns.despine(left=True, bottom=True)
style_axes(ax)
save_fig(fig, "nb_faers_severity")
plt.show()

### 3.2 Population Allele Frequencies: Saudi vs. Global

Saudi minor allele frequencies (`saudi_pgx_ngs/S1_table.xlsx`) against the Gulf/Middle-East-wide (`GME_ALL`) reference already included in that table, for the three warfarin pharmacogene variants it covers.

In [ ]:
s1 = load_cached(BASE_DIR / "usable/saudi_pgx_ngs/S1_table.xlsx")
s1 = s1.assign(Gene=s1["Gene"].ffill())

variants = {
    "rs1799853": ("CYP2C9", "*2"),
    "rs1057910": ("CYP2C9", "*3"),
    "rs2108622": ("CYP4F2", "*3"),
}
rows = []
for rsid, (gene, allele) in variants.items():
    match = s1[s1["Defining variants"] == rsid]
    if not match.empty:
        r = match.iloc[0]
        rows.append({"Variant": f"{gene}{allele} ({rsid})", "Saudi MAF": r["MAF  Saudi"], "GME_ALL": r["GME_ALL "]})
saudi_gme = pd.DataFrame(rows)
print(saudi_gme)

# rs9923231 (VKORC1 -1639G>A) has no Saudi comparator in S1_table.xlsx — confirmed, not omitted by accident.
missing = s1[s1["Defining variants"] == "rs9923231"]
print(f"\nrs9923231 (VKORC1 -1639G>A) rows in S1_table: {len(missing)} — no Saudi MAF available for this variant.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)
x = np.arange(len(saudi_gme))
width = 0.35
pal = sns.cubehelix_palette(2, rot=-0.25, light=0.7)
b1 = ax.bar(x - width / 2, saudi_gme["Saudi MAF"], width, label="Saudi", color=pal[0])
b2 = ax.bar(x + width / 2, saudi_gme["GME_ALL"], width, label="Gulf/Middle East (GME_ALL)", color=pal[1])
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005, f"{h:.2f}", ha="center", va="bottom", color="dimgrey", size=10)
ax.set_xticks(x)
ax.set_xticklabels(saudi_gme["Variant"])
ax.set_ylabel("Minor allele frequency")
ax.set_title("Warfarin pharmacogene allele frequencies: Saudi vs. regional reference", loc="left", fontsize=13, pad=28)
ax.text(0, 1.0, "VKORC1 -1639G>A (rs9923231) omitted — no Saudi comparator in this dataset", transform=ax.transAxes,
        fontsize=10, color="dimgrey", va="bottom")
ax.legend(frameon=True, facecolor="white", framealpha=0.8, edgecolor="lightgrey", labelcolor="dimgrey")
ax.grid(alpha=0.7, linewidth=1, axis="y")
sns.despine(left=True, bottom=True)
style_axes(ax)
save_fig(fig, "nb_saudi_vs_gme_allele_freq")
plt.show()

### 3.3 MIMIC-IV Warfarin Cohort Snapshot

No pre-filtered warfarin file exists for MIMIC-IV (unlike MIMIC-III) — filtered here by drug-name match, restricted to the demo cohort's subjects.

In [ ]:
mimic4_rx = load_cached(BASE_DIR / "usable/mimic4_demo/icu_hosp/2.2/hosp/prescriptions.xlsx")
demo_subjects = load_cached(BASE_DIR / "usable/mimic4_demo/icu_hosp/2.2/demo_subject_id.xlsx")

mimic4_warfarin = mimic4_rx[
    mimic4_rx["drug"].astype(str).str.contains("warfarin", case=False, na=False)
    & mimic4_rx["subject_id"].isin(demo_subjects.iloc[:, 0])
].copy()
mimic4_warfarin["dose_val_rx"] = pd.to_numeric(mimic4_warfarin["dose_val_rx"], errors="coerce")

mimic4_summary = pd.DataFrame([{
    "Dataset": "MIMIC-IV demo",
    "Warfarin orders": len(mimic4_warfarin),
    "Unique patients": mimic4_warfarin["subject_id"].nunique(),
    "Dose range (mg)": f"{mimic4_warfarin['dose_val_rx'].min():.0f}-{mimic4_warfarin['dose_val_rx'].max():.0f}"
        if mimic4_warfarin["dose_val_rx"].notna().any() else "n/a",
}])
mimic4_summary.style.hide(axis="index").set_caption("MIMIC-IV demo — warfarin cohort snapshot")

### 3.4 eICU-CRD Warfarin Order Count

Same treatment as 3.3, for the eICU-CRD demo's medication table.

In [ ]:
eicu_med = load_cached(BASE_DIR / "usable/eicu_crd_demo/2.0.1/medication.xlsx")
eicu_warfarin = eicu_med[eicu_med["drugname"].astype(str).str.contains("warfarin", case=False, na=False)]

eicu_summary = pd.DataFrame([{
    "Dataset": "eICU-CRD demo",
    "Warfarin orders": len(eicu_warfarin),
    "Unique patients": eicu_warfarin["patientunitstayid"].nunique(),
}])
eicu_summary.style.hide(axis="index").set_caption("eICU-CRD demo — warfarin order snapshot")

### 3.5 Most-Mentioned Pharmacogene

Which warfarin pharmacogene does the systematic review's own synthesis discuss most? Counted directly from the manuscript text (`MDPI_Article/main.tex`), word-boundary matched so substring hits (e.g. "process" containing "proc") don't inflate counts.

In [ ]:
import re

MANUSCRIPT_PATH = BASE_DIR.resolve().parent / "MDPI_Article" / "main.tex"
manuscript_text = MANUSCRIPT_PATH.read_text()

CANDIDATE_GENES = ["VKORC1", "CYP2C9", "CYP4F2", "CYP2C19", "CYP3A5", "CYP1A2", "GGCX", "CYP3A4", "ABCB1", "EPHX1"]
gene_mentions = pd.Series({
    gene: len(re.findall(rf"\b{gene}\b", manuscript_text, flags=re.IGNORECASE))
    for gene in CANDIDATE_GENES
}).sort_values(ascending=False)
gene_mentions = gene_mentions[gene_mentions > 0]

print("Most-mentioned warfarin pharmacogene in the review manuscript:")
print(gene_mentions)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5), dpi=150)
pal = sns.cubehelix_palette(len(gene_mentions), rot=-0.25, light=0.7)
ax.barh(gene_mentions.index[::-1], gene_mentions.values[::-1], color=pal[::-1])
for i, v in enumerate(gene_mentions.values[::-1]):
    ax.text(v + 0.15, i, str(v), va="center", color="dimgrey", weight="semibold", size=11)
ax.set_xlabel("Mentions in manuscript text")
tie_note = " (tied)" if gene_mentions.iloc[0] == gene_mentions.iloc[1] else ""
ax.set_title(f"Most-discussed warfarin pharmacogene{tie_note} in this systematic review", loc="left", fontsize=13, pad=28)
ax.text(0, 1.0, "Word-boundary count across the review manuscript (main.tex)", transform=ax.transAxes,
        fontsize=10, color="dimgrey", va="bottom")
ax.axvline(x=0, color="lightgrey", linewidth=0.8, zorder=0)
ax.grid(False)
sns.despine(left=True, bottom=True)
style_axes(ax)
save_fig(fig, "nb_gene_mention_frequency")
plt.show()

### 3.6 Population Variability: VKORC1 -1639G>A

The manuscript's cited GWAS work (Elrouby et al. 2022) found *VKORC1* rs9923231 dose to range from 27 mg/week (AA) to 39 mg/week (AG) to 52 mg/week (GG) — a >2x swing driven by a single variant. How much does the frequency of that variant itself vary by ancestry? All populations in the 1000 Genomes/gnomAD export for this SNP, ranked.

In [ ]:
rs9923231 = load_cached(BASE_DIR / "usable/gme_1000genomes_freq/rs9923231.xlsx", sheet="population_frequencies")

# This export reports rs9923231 on the minus strand (alleles C/T, not the clinical G/A convention).
# Verified against known population genetics: T is ~88% in East Asian populations and ~5% in African
# populations here, matching the well-documented clinical "A" (low-dose) allele pattern -- so T is
# used as the clinical-A-equivalent allele below.
freq_a = rs9923231[rs9923231["allele"] == "T"].sort_values("frequency", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(7, 7), dpi=150)
pal = sns.cubehelix_palette(len(freq_a), rot=-0.25, light=0.7)
ax.barh(freq_a["population"][::-1], freq_a["frequency"][::-1], color=pal[::-1])
for i, v in enumerate(freq_a["frequency"].values[::-1]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center", color="dimgrey", weight="semibold", size=10)
ax.set_xlabel("VKORC1 -1639 low-dose ('A') allele frequency")
ax.set_title("Population variability in a dose-determining variant (rs9923231)", loc="left", fontsize=12, pad=28)
ax.text(0, 1.0, "Top 15 populations by frequency, 1000 Genomes / gnomAD", transform=ax.transAxes,
        fontsize=10, color="dimgrey", va="bottom")
ax.axvline(x=0, color="lightgrey", linewidth=0.8, zorder=0)
ax.grid(False)
sns.despine(left=True, bottom=True)
style_axes(ax)
save_fig(fig, "nb_vkorc1_population_variability")
plt.show()

### 3.7 Our SHAP Ranking vs. the One Published SHAP Application

The manuscript's central explainability finding: the only other SHAP application identified in the included literature (Guo et al., cited in Fulop et al. 2025) found genetics absent from its top-ranked features, the opposite of what our reanalysis (Section 2.3) found. Side-by-side, using the relative feature-contribution percentages each analysis reports.

In [ ]:
shap_comparison = pd.DataFrame([
    {"Feature category": "Genetic (CYP2C9/VKORC1)", "This reanalysis (%)": genetic_share_pct, "Guo et al., published (%)": 0},
    {"Feature category": "Previous INR values", "This reanalysis (%)": None, "Guo et al., published (%)": 42},
    {"Feature category": "Age", "This reanalysis (%)": None, "Guo et al., published (%)": 18},
    {"Feature category": "Concurrent medications", "This reanalysis (%)": None, "Guo et al., published (%)": 15},
    {"Feature category": "Body weight", "This reanalysis (%)": None, "Guo et al., published (%)": 12},
    {"Feature category": "Clinical/demographic (other)", "This reanalysis (%)": 100 - genetic_share_pct, "Guo et al., published (%)": None},
])
print("Guo et al. had no longitudinal INR history feature available in our cross-sectional cohort; "
      "the most parsimonious reading (per the manuscript) is that genetics fills that predictive role when INR history is absent.")

(shap_comparison.style.hide(axis="index")
    .set_caption("SHAP-ranked feature contribution: this reanalysis vs. the literature's one SHAP application")
    .format({"This reanalysis (%)": "{:.1f}", "Guo et al., published (%)": "{:.0f}"}, na_rep="—")
    .set_table_styles([{"selector": "th", "props": [("font-weight", "bold")]}]))

In [ ]:
plot_df = shap_comparison.set_index("Feature category").fillna(0)
categories = plot_df.index.tolist()
y = np.arange(len(categories))
height = 0.35
pal = sns.cubehelix_palette(2, rot=-0.25, light=0.7)

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
ax.barh(y + height / 2, plot_df["This reanalysis (%)"], height, label="This reanalysis", color=pal[0])
ax.barh(y - height / 2, plot_df["Guo et al., published (%)"], height, label="Guo et al., published", color=pal[1])
for yi, (a, b) in enumerate(zip(plot_df["This reanalysis (%)"], plot_df["Guo et al., published (%)"])):
    if a > 0:
        ax.text(a + 1, yi + height / 2, f"{a:.0f}%", va="center", color="dimgrey", weight="semibold", size=10)
    if b > 0:
        ax.text(b + 1, yi - height / 2, f"{b:.0f}%", va="center", color="dimgrey", weight="semibold", size=10)
ax.set_yticks(y)
ax.set_yticklabels(categories)
ax.set_xlabel("Relative feature contribution (%)")
ax.set_title("This reanalysis's SHAP ranking vs. the literature's one SHAP application", loc="left", fontsize=12, pad=28)
ax.text(0, 1.0, "Guo et al., cited in Fulop et al. 2025 -- opposite ranking: genetics absent from their top features",
        transform=ax.transAxes, fontsize=10, color="dimgrey", va="bottom")
ax.set_xlim(0, plot_df.to_numpy().max() * 1.2)
ax.legend(frameon=True, facecolor="white", framealpha=0.8, edgecolor="lightgrey", labelcolor="dimgrey",
          loc="center left", bbox_to_anchor=(1.02, 0.5))
ax.axvline(x=0, color="lightgrey", linewidth=0.8, zorder=0)
ax.grid(False)
sns.despine(left=True, bottom=True)
style_axes(ax)
save_fig(fig, "nb_shap_comparison")
plt.show()

## 4. Summary

**Reproduced from the original notebook (Section 2):** IWPC algorithm benchmark, clinical-vs-genetic feature
comparison, SHAP importance, VKORC1/CYP2C9 genotype-dose boxplots, MIMIC-III warfarin dose/INR reanalysis, NOMEN PK
profiles, and the MIPD simulation dosing-arm reanalysis (values reproduced from the cached pipeline output — see the
methodology note in Section 2.7; recomputing MAE from scratch requires the original mechanistic ODE simulation, not
a flat-file operation).

**Newly added (Section 3):** FAERS adverse event severity breakdown, Saudi-vs-regional allele frequency comparison,
MIMIC-IV and eICU-CRD warfarin cohort snapshots, most-mentioned pharmacogene in the review manuscript, population
variability in the VKORC1 -1639G>A dose-determining variant, and a side-by-side comparison of this reanalysis's SHAP
ranking against the one other published SHAP application in the included literature.

**Intentionally skipped:**
- **`pharmgkb_allele_tables` CYP2C9/CYP4F2 definition tables** — merged-cell header layout, not cleanly parseable;
  kept as reference-only entries in the inventory.
- **`nomen_pk` covariate-importance benchmark matrices** (`SI-AIEP-paper/checkpoints/*.xlsx`) — unlabeled rows/columns;
  semantics unrecoverable without the original fitting code.

**Housekeeping:** regenerated figures are written to `./figures/` as matched PDF+PNG pairs, then synced into
`Systematic Review /files/images/` (the folder `MDPI_Article/images` symlinks to) so the manuscript's `\includegraphics`
references stay current.
